In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev",
    choices=["fq_dev", "fq_test", "fq_prod"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="POSIST",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="sales",
    choices=["discount", "sales", "cost"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

# Get external location URLs
bronze_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_bronze`"
).select("url").collect()[0][0]

silver_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_silver`"
).select("url").collect()[0][0]

gold_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_gold`"
).select("url").collect()[0][0]

checkpoint = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_checkpoint`"
).select("url").collect()[0][0]

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_staging`"
).select("url").collect()[0][0]

print(f"Environment: {environment}")
print(f"Source: {source}")
print(f"Domain: {domain}")

In [0]:
%sql
select * from fq_dev_catalog.silver.sales_items_silver_2 where deployment_name = 'ALBAIK - SQ DB14 - ADNOC DUBAI HILLS - 1005014' and business_date = '2026-01-01' limit 1

In [0]:
%sql
-- alter table fq_dev_catalog.gold.sales_item_wise add columns (unit_cost DECIMAL(18, 3));

In [0]:
%sql
CREATE TABLE IF NOT EXISTS fq_dev_catalog.gold.sales_item_wise (
    business_date DATE,
    deployment_name STRING,
    item_code STRING,
    item_name STRING,
    order_type STRING,
    tab_name STRING,
    quantity BIGINT,
    subtotal DECIMAL(18,3),
    tax_amount DECIMAL(18,3),
    gross_amount DECIMAL(18,3),
    category STRING,
    super_category STRING,
    created_ts TIMESTAMP, 
    last_modified_ts TIMESTAMP,
    unit_cost DECIMAL(18, 3)
)
PARTITIONED BY ( -- Low cardinality columns first
    deployment_name
)
TBLPROPERTIES (
    delta.columnMapping.mode = 'name',
    delta.enableChangeDataFeed = true,
    delta.autoOptimize.optimizeWrite = true,
    delta.autoOptimize.autoCompact = true
);

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col, sum, current_timestamp
import time

def upsert_sales_item_wise(microBatchDF, batchId):
    windowSpec = Window.partitionBy(
        "ng_id"
    ).orderBy(col("_commit_version").desc())
    deduped = (
        microBatchDF.withColumn("rank", dense_rank().over(windowSpec))
        .filter("rank = 1 and _change_type != 'update_preimage'")
        .drop("rank", "_commit_version")
    )

    active = deduped.filter(col("_change_type") != "delete")

    gold_df = (
        active.groupBy(
            "order_type", "tab_name", "supercategory_name", "deployment_name", "category_name", "business_date", "item_upccode", "item_name"
        )
        .agg(
            sum("quantity").alias("quantity"),
            sum("net_item_amount").alias("subtotal").cast("decimal(18,3)"),
            sum("tax_amount").alias("tax_amount").cast("decimal(18,3)"),
            sum("gross_item_amount").alias("gross_amount").cast("decimal(18,3)")
        ).withColumnRenamed("item_upccode", "item_code") \
        .withColumnRenamed("category_name", "category") \
        .withColumnRenamed("supercategory_name", "super_category")
        .withColumn("created_ts", current_timestamp())
        .withColumn("last_modified_ts", current_timestamp())
    )

    (DeltaTable.forName(spark, "fq_dev_catalog.gold.sales_item_wise").alias("target")
    .merge(
        gold_df.alias("source"),
        "source.business_date = target.business_date AND "
        "source.deployment_name = target.deployment_name AND "
        "source.item_code = target.item_code AND "
        "source.item_name = target.item_name AND "
        "source.order_type = target.order_type AND "
        "source.tab_name = target.tab_name AND "
        "source.category = target.category AND " 
        "source.super_category = target.super_category"
    )
    .whenMatchedUpdate(
        set={
            "quantity": "source.quantity",
            "subtotal": "source.subtotal",
            "tax_amount": "source.tax_amount",
            "gross_amount": "source.gross_amount",
            "last_modified_ts": "source.last_modified_ts"
        }
    )
    .whenNotMatchedInsert(
        values={
            "business_date": "source.business_date",
            "deployment_name": "source.deployment_name",
            "item_code": "source.item_code",
            "item_name": "source.item_name",
            'order_type': "source.order_type",
            'tab_name': "source.tab_name",
            "category": "source.category",
            "super_category": "source.super_category",
            "quantity": "source.quantity",
            "subtotal": "source.subtotal",
            "tax_amount": "source.tax_amount",
            "gross_amount": "source.gross_amount",
            "created_ts": "source.created_ts",
            "last_modified_ts": "source.last_modified_ts"
        }
    )
    .execute())

(spark.readStream
    .option("readChangeData", "true")
    .option("startingVersion", 58)
    .option("schemaTrackingLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpoint_sales_item_wise/sales_items_schema_tracking') # schema-evolution
    .table("fq_dev_catalog.silver.sales_items_silver_2")
    .writeStream
    .foreachBatch(upsert_sales_item_wise)
    .option('mergeSchema', True) # schema-evolution
    .option("checkpointLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpoint_sales_item_wise')
    .trigger(availableNow=True)
    .start()
).awaitTermination()

print('Stream stopped')

In [0]:
%sql
SELECT 
  count(distinct super_category) as super_category_cardinality,
  count(distinct deployment_name) as deployment_name_cardinality,
  count(distinct category) as category_cardinality,
  count(distinct business_date) as business_date_cardinality,
  count(distinct item_code) as item_code_cardinality,
  count(distinct item_name) as item_name_cardinality,
  COUNT(*) AS total_rows
FROM fq_dev_catalog.gold.sales_item_wise;

In [0]:
%sql
OPTIMIZE fq_dev_catalog.gold.sales_item_wise ZORDER BY (item_name, item_code, business_date); -- high cardinality non-partitioned column first

In [0]:
%sql
select * from fq_dev_catalog.gold.sales_item_wise limit 1--where deployment_name = 'ALBAIK - SQ DB14 - ADNOC DUBAI HILLS - 1005014' and business_date = '2026-01-01'

In [0]:
for query in spark.streams.active:
    query.stop()